# Fake It Till You Make It — Colab Quickstart
Detect AI-generated speech and measure seen-vs-unseen generalization.

**Runtime → Change runtime type → GPU (T4 to start, L4 for the full runs).**

Timeline on a T4: Kaggle data download+extract ≈ 20–30 min (once per session), subset sanity run ≈ 20 min, full wav2vec2+LoRA run ≈ 3–5 h (auto-resumes if the session dies — just rerun the cell).

In [ ]:
# Clone (idempotent: safe to re-run without nesting the repo inside itself).
%cd /content
![ -d fake-it-till-you-make-it ] && rm -rf fake-it-till-you-make-it
!git clone https://github.com/taha-tahaa/fake-it-till-you-make-it.git
%cd /content/fake-it-till-you-make-it
!pip install -q -r requirements.txt
# Colab ships an old torchao that newer peft rejects on import; we don't use it.
!pip uninstall -y torchao >/dev/null 2>&1 || true

In [ ]:
# CRASH-PROOFING: point runs/ at Google Drive so finished experiments survive a
# Colab VM recycling (which wipes the VM disk). On a fresh VM, re-running the
# training cell auto-SKIPS any experiment whose metrics are already on Drive.
from google.colab import drive
import os, glob, json, shutil
drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dlproj/runs'
os.makedirs(persist, exist_ok=True)
if os.path.islink('runs'):
    os.remove('runs')
elif os.path.exists('runs'):
    shutil.rmtree('runs')
os.symlink(persist, 'runs')
print('runs/ ->', os.readlink('runs'))
done = [os.path.basename(os.path.dirname(m)) for m in glob.glob('runs/*/metrics.json')
        if 'eval_eer_unseen' in json.load(open(m))]
print('already complete (will be skipped on re-run):', done or 'none yet')

### Kaggle credentials (one-time)
The dataset comes from the Kaggle mirror `awsaf49/asvpoof-2019-dataset`, fetched via `kagglehub`. Get a token at **kaggle.com → Settings → API → Create New Token**, then paste the bearer token (starts with `KGAT_`) below when prompted.

In [ ]:
import os
from pathlib import Path
kdir = Path.home() / '.kaggle'
kdir.mkdir(parents=True, exist_ok=True)
token_file = kdir / 'access_token'
if not token_file.exists():
    tok = input('Paste your Kaggle token (KGAT_...): ').strip()
    token_file.write_text(tok)
    os.chmod(token_file, 0o600)
print('credentials ready:', token_file.exists())

In [ ]:
# ASVspoof 2019 LA (~23.6 GB) from the Kaggle mirror. Downloads to the fast VM disk,
# extracts, normalizes the folder layout, and verifies protocols vs files.
!python scripts/download_data.py --out ./data

In [ ]:
# Sanity check everything before burning GPU-hours (metrics, protocol parser, models).
!python -m pytest tests -q
!RUN_SLOW=1 python -m pytest tests/test_models.py -q

In [ ]:
# Fast-iteration subset (~20 min): confirms the whole pipeline learns before the full runs.
!python scripts/make_subset.py --root ./data
!python -m src.train --config configs/baseline_lcnn_lfcc.yaml --subset

In [ ]:
# FULL wav2vec2 experiments (the heavy half — needs Colab's GPU).
# The 3 LCNN baselines run in parallel on the local machine, so we skip them here.
# Priority order: main LoRA model first, then ablations. Resumable across disconnects.
!python scripts/run_all.py --only wav2vec2_lora wav2vec2_probe wav2vec2_dora wav2vec2_lora_noaug wav2vec2_lora_ocsoftmax

In [ ]:
# runs/ already persists to Drive (crash-proofing cell above), so just copy the
# generated figures + results table over too.
!mkdir -p /content/drive/MyDrive/dlproj && cp -r results /content/drive/MyDrive/dlproj/ 2>/dev/null
print('figures + table saved to Drive/dlproj/results ; checkpoints already in Drive/dlproj/runs')

In [ ]:
# Save checkpoints + scores + metrics to Drive so nothing is lost when the VM dies.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/dlproj && cp -r runs results /content/drive/MyDrive/dlproj/